# 43. Learning-rate sweep for the revised stream

This is phase 2a of the revised plasticity study: runs 1 to 5 of thirteen. It
picks the learning rate that every later run will use.

## Why a sweep comes first

The earlier version of this work took the learning rate from the original
hypernetwork paper and never tuned it. Accuracy was low and the spread between
seeds was wide, which makes every later comparison hard to read: a difference
between two runs could be a real effect or could be the noise floor. The
methodology now tunes before measuring anything, and reports the rejected
values alongside the chosen one.

## What this notebook does

Five short runs, one per candidate learning rate, each learning only the first
three tasks of `order_A`. Everything else is held fixed: seed 0, five epochs
per task, batch size 64, ResNet50 written by the generator, hyperfan-in
initialization. The value with the highest mean accuracy over those three
tasks wins. The short stream is what keeps the sweep cheap.

The candidates are centred on the current default of 1e-3: 3e-4, 5e-4, 1e-3,
2e-3 and 4e-3. That is a third of the default, half of it, the default itself,
double and four times.

## What this notebook does not do

It reads no plasticity indicator and writes nothing to R2. The sweep only needs
accuracy. Boundaries, fixed image sets and the held-out probe belong to the
nine-task runs, which come after this and only if this one settles on a value.

## The gate after this

The chosen value then has to survive five seeds of the full nine-task stream,
finishing within 3 percentage points of each other on mean accuracy. If it does
not, the setting is not good enough to measure against and the tuning goes
further. Nothing in runs 11 to 13 starts before that gate passes.

## Before you run

Pick a GPU runtime. A task now holds twenty classes at 500 training images
each, which is 10,000 images per epoch, twice the old task size. Section 4
prints the wall clock of the first run and projects the rest, so you can stop
there if the projection is worse than the budget.

A run that is interrupted picks itself up. Each value writes its own output
directory with a resumable checkpoint, so re-running section 4 after a
disconnect continues where the session died rather than starting over.

## What to share afterwards

The table that section 5 prints, and the `sweep_summary.json` it writes. Both
hold every value, including the rejected ones.

Notebook 43 was once sketched for a post-unlearn boundary model of the earlier
study. That study cannot be continued across the new class split, so the number
is reused here.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This notebook runs at the current main, not at a pinned revision. The
# ten-group class split landed on 12 November 2026, and a pinned checkout from
# before that date would still cut twenty groups of ten.
if IN_COLAB:
    REPO = Path("/content/path-sweep")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    # The new split's file, not task_partition.json. A checkout that predates
    # the split has only the old file and would load a different experiment.
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError(
            "Open this notebook from a PATH checkout that has the ten-group split.")

if importlib.util.find_spec("tqdm") is None and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "tqdm"], check=True)
sys.path.insert(0, str(REPO))

import torch
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.experiments import run_experiment
from uncle.tasks import CLASSES_PER_TASK, TASK_COUNT

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print(f"Split: {TASK_COUNT} groups of {CLASSES_PER_TASK} classes "
      f"| chance accuracy {100 / CLASSES_PER_TASK:.0f}%")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This notebook trains; switch to a GPU runtime before section 4.")

## 2. Settings

In [ ]:
# The five candidate learning rates, centred on the current default of 1e-3.
# Every one of them is reported at the end, chosen or rejected, because the
# methodology says the tuning runs are reported in full. Dropping a value from
# this list after seeing its result would turn the sweep into a story about
# the winner.
LEARNING_RATES = (3e-4, 5e-4, 1e-3, 2e-3, 4e-3)

# The sweep learns the first three tasks of order_A and stops. Three tasks are
# enough to separate the candidates on accuracy, and nine would cost three
# times as much for a decision that does not need them.
ORDER = "order_A"
TASKS_IN_SWEEP = 3

# One seed for the sweep. Seed spread is what the five-seed confirmation run
# measures, after a value has been picked. Measuring it here would multiply the
# sweep by five to answer a question the sweep does not ask.
SEED = 0

# Held fixed across all five runs, so the learning rate is the only thing that
# differs between them.
#
#   epochs           5 per task, the setting the earlier runs used
#   batch_size       64, likewise
#   backbone         ResNet50, written by the generator (methodology 3.2)
#   beta             0.01, the UnCLe paper's Tiny ImageNet value, supplied by
#                    study_defaults. This is the term that holds earlier tasks
#                    still. It is tuned only if the learning-rate sweep alone
#                    misses its target.
#   initialization   hyperfan_in, which every run of the earlier study used.
#                    "legacy" exists so old checkpoints keep their output
#                    scales, and is not wanted for a clean re-run.
EPOCHS = 5
BATCH_SIZE = 64
INITIALIZATION = "hyperfan_in"

DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
# Where the runs write. On Colab the runtime is recycled after a few hours
# and /content goes with it, so a sweep longer than one session would lose
# every checkpoint and start from the top. If Google Drive is mounted the
# outputs go there instead and a new session resumes where the old one died.
# Mount it in a cell above this one, before running the sweep:
#
#     from google.colab import drive
#     drive.mount("/content/drive")
#
# The dataset deliberately stays on /content. Training reads it constantly,
# Drive is slow for that, and re-downloading 240MB is cheap by comparison.
DRIVE = Path("/content/drive/MyDrive")
if IN_COLAB and DRIVE.exists():
    OUTPUT_ROOT = DRIVE / "plasticity-lr-sweep"
elif IN_COLAB:
    OUTPUT_ROOT = Path("/content/sweep")
else:
    OUTPUT_ROOT = REPO / "outputs/lr-sweep"
SUMMARY_PATH = OUTPUT_ROOT / "sweep_summary.json"

# Every run writes a resumable checkpoint, about 220 MB for ResNet50 at 200
# chunks, overwritten in place. Five values means five of them. On a Colab
# runtime that is local disk and disappears with the session, which is fine:
# it only has to survive a disconnect inside one session.
KEEP_CHECKPOINTS = True

order_requests = study_defaults(ORDER)["requests"]
sweep_requests = order_requests[:TASKS_IN_SWEEP]
print("Order:", ORDER, "| full stream:", " ".join(task for _, task in order_requests))
print("This sweep learns:", " ".join(task for _, task in sweep_requests))
print("Held out as the probe group, never trained:", PROBE_TASK)
print("Candidates:", ", ".join(f"{value:g}" for value in LEARNING_RATES))
print("Outputs:", OUTPUT_ROOT)
if IN_COLAB and not DRIVE.exists():
    print("  Drive is not mounted, so these are lost when the runtime recycles.")
    print("  Mount it and re-run this cell if the sweep will outlast one session.")

## 3. Build the configurations

In [ ]:
# One Config per candidate. `study_defaults` supplies the dataset, the task
# list, the request order, beta and the backbone, so the only thing that
# differs between these five is the learning rate.
#
# `tasks` holds all ten groups even though three are learned here. Nothing
# requests the others, so nothing trains on them, and run_experiment indexes
# only the tasks its requests name.
configs = {}
for value in LEARNING_RATES:
    settings = study_defaults(ORDER)
    settings["requests"] = sweep_requests
    configs[value] = Config(
        **settings,
        seed=SEED,
        learning_rate=value,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        initialization=INITIALIZATION,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

example = configs[LEARNING_RATES[0]]
print("classes per task:", example.classes_per_task,
      "| backbone:", example.backbone,
      "| beta:", example.beta,
      "| epochs:", example.epochs,
      "| batch:", example.batch_size)
print("requests:", " ".join(f"{action[0].upper()}{task}"
                            for action, task in example.requests))
print()
print("runs to do:")
for value in LEARNING_RATES:
    print(f"  sweep_lr{value:g}   learning rate {value:g}")

## 4. Run the sweep

In [ ]:
from statistics import mean

# run_experiment writes a checkpoint after every request and picks it up on a
# second call, so an interrupted session continues at the task it died on
# rather than at the top. Re-running this cell after a disconnect is the
# intended way to finish.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
results = {}

for index, value in enumerate(LEARNING_RATES, start=1):
    name = f"sweep_lr{value:g}"
    output = OUTPUT_ROOT / name
    print()
    print("=" * 70)
    print(f"[{index}/{len(LEARNING_RATES)}] {name}")
    print("=" * 70)

    began = time.perf_counter()
    run = run_experiment(
        config=configs[value],
        root=DATA_ROOT,
        download=True,
        output=output,
        checkpoint=KEEP_CHECKPOINTS,
        resume=True,
        verbose=True,
        progress=True,
    )
    took = time.perf_counter() - began

    # The last record holds every task learned so far, so its "after" block is
    # the accuracy of all three at the end of the short stream. That mean is
    # what picks the winner.
    final = run["history"][-1]["after"]
    learned = [task for _, task in sweep_requests]
    per_task = {task: final[task] for task in learned}
    results[value] = {
        "run": name,
        "learning_rate": value,
        "seed": SEED,
        "order": ORDER,
        "tasks": learned,
        "accuracy_per_task": per_task,
        "mean_accuracy": mean(per_task.values()),
        "wall_clock_seconds": took,
        "code_revision": HEAD,
    }

    print()
    print(f"{name}: mean accuracy over {len(learned)} tasks "
          f"{results[value]['mean_accuracy']:.2f}%   "
          + "  ".join(f"task {t} {a:.1f}%" for t, a in per_task.items()))
    print(f"wall clock {took / 60:.1f} minutes for {len(learned)} tasks, "
          f"about {took / 60 / len(learned):.1f} minutes per task")

    # The projection, printed after the first value rather than at the end, so
    # an unaffordable sweep can be stopped here.
    #
    # A task costs more the later it arrives, because the term that holds
    # earlier tasks still has one more task to hold at every step. On the
    # first L4 run the three tasks took 0.244, 0.361 and 0.470 seconds per
    # step. Multiplying a flat per-task mean by 63, which this cell used to
    # do, understated the seven nine-task runs by roughly half, so the line
    # below is fitted to the rise and extended to nine positions instead.
    if index == 1:
        seconds = [row["seconds"] for row in run["costs"]]
        count = len(seconds)
        # Least squares through (position, seconds). Two points would give a
        # slope with nothing to check it against, so with fewer than three
        # tasks the fit is dropped and the flat mean is used and labelled.
        mean_x = (count - 1) / 2
        mean_y = sum(seconds) / count
        spread = sum((i - mean_x) ** 2 for i in range(count))
        slope = (sum((i - mean_x) * (y - mean_y) for i, y in enumerate(seconds))
                 / spread) if spread and count >= 3 else 0.0
        intercept = mean_y - slope * mean_x
        nine_task_hours = sum(intercept + slope * i for i in range(9)) / 3600

        print()
        print("Per task so far: "
              + " ".join(f"{s / 60:.1f}" for s in seconds) + " minutes"
              + (f", rising about {slope / 60:.1f} min per extra task held"
                 if slope else " (flat: too few tasks to fit a rise)"))
        print(f"  rest of the sweep     "
              f"{sum(seconds) * (len(LEARNING_RATES) - 1) / 3600:.1f} hours")
        print(f"  one nine-task run     {nine_task_hours:.1f} hours")
        print(f"  the 7 nine-task runs  {nine_task_hours * 7:.1f} hours")
        print("  Stop here if that is more than the budget allows.")
        print("  The fit rests on three points, so treat it as a floor.")

## 5. The result, and what to share

In [ ]:
# Every value, in the order they were tried, with the winner marked. The
# rejected values stay in the table and in the file: a sweep reported as a
# single chosen number cannot be checked by anyone.
ranked = sorted(results.values(), key=lambda row: row["mean_accuracy"], reverse=True)
best = ranked[0] if ranked else None

print(f"{'learning rate':>14}{'mean accuracy':>16}{'per task':>24}{'minutes':>10}")
for value in LEARNING_RATES:
    if value not in results:
        print(f"{value:>14g}{'not run':>16}")
        continue
    row = results[value]
    mark = "   <- chosen" if best and value == best["learning_rate"] else ""
    per_task = " ".join(f"{a:5.1f}" for a in row["accuracy_per_task"].values())
    print(f"{value:>14g}{row['mean_accuracy']:>15.2f}%{per_task:>24}"
          f"{row['wall_clock_seconds'] / 60:>10.1f}{mark}")

if best:
    chance = 100 / example.classes_per_task
    print()
    print(f"Chosen learning rate: {best['learning_rate']:g}, mean accuracy "
          f"{best['mean_accuracy']:.2f}% over {len(best['tasks'])} tasks of "
          f"{example.classes_per_task} classes.")
    print(f"Chance accuracy is {chance:.0f}%. A winner near chance means the "
          "sweep found nothing worth confirming, and the range of candidates "
          "needs widening before the nine-task runs start.")

summary = {
    "study": "plasticity_lr_sweep",
    "phase": "2a",
    "code_revision": HEAD,
    "order": ORDER,
    "seed": SEED,
    "tasks_in_sweep": TASKS_IN_SWEEP,
    "classes_per_task": example.classes_per_task,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "beta": example.beta,
    "backbone": example.backbone,
    "initialization": INITIALIZATION,
    "candidates": [f"{value:g}" for value in LEARNING_RATES],
    "results": [results[value] for value in LEARNING_RATES if value in results],
    "chosen": best,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print()
print("Written:", SUMMARY_PATH)
print("Share that file and the table above. It holds the rejected values too.")

## 6. What comes next

Runs 6 to 10 take the chosen value to the full nine-task stream, five seeds,
with the plasticity indicators attached and boundaries saved to R2. That is a
different notebook and a different kind of run: it measures at ten boundaries
and keeps models, where this one only reports accuracy.

The gate before it is the one in the methodology. Five seeds must finish within
3 percentage points of each other on mean accuracy over the nine tasks. If they
do not, the chosen value is not good enough to measure against, and epochs per
task and beta are tuned next rather than reported on top of.